# Trực quan hóa kết quả đánh giá các mô hình SpikeGPT và GPT2

Notebook này nhằm mục đích:
1. Chứng minh **HeadQK** cải thiện hiệu suất bằng cách so sánh SpikeGPT from scratch (epoch 78) có và không có HeadQK.
2. Chứng minh **Finetune** cải thiện hiệu suất bằng cách so sánh SpikeGPT from scratch và SpikeGPT finetune (epoch 220).
3. Chứng minh **SpikeGPT vẫn còn khoảng cách so với Transformer** bằng cách so sánh SpikeGPT (phiên bản tốt nhất) với GPT2-Small và GPT2-Medium.

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

def parse_metrics(filepath):
    metrics = {}
    if not os.path.exists(filepath):
        print(f"Warning: {filepath} not found.")
        return metrics
    
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
        
        m = re.search(r'Sinh JSON hợp lệ \(Valid JSON\)\s*:\s*([\d.]+)%', content)
        if m: metrics['Valid JSON (%)'] = float(m.group(1))
            
        m = re.search(r'Chọn đúng Tool \(Intent Acc\)\s*:\s*([\d.]+)%', content)
        if m: metrics['Intent Accuracy (%)'] = float(m.group(1))
            
        m = re.search(r'Copy đúng ID \(Args Exact Match\)\s*:\s*([\d.]+)%', content)
        if m: metrics['Args Exact Match (%)'] = float(m.group(1))
            
        m = re.search(r'Precision \(Weighted\):\s*([\d.]+)', content)
        if m: metrics['Precision'] = float(m.group(1))
            
        m = re.search(r'Recall \(Weighted\)\s*:\s*([\d.]+)', content)
        if m: metrics['Recall'] = float(m.group(1))
            
        m = re.search(r'F1 Score \(Weighted\)\s*:\s*([\d.]+)', content)
        if m: metrics['F1 Score'] = float(m.group(1))
            
    return metrics

In [ ]:
base_dir = r"d:\Study\HK6\CS338-NhanDang\CS338\eval_results"
models = {
    "SpikeGPT_78_NoHeadQK": os.path.join(base_dir, "78_NoHeadQK", "metrics_report.txt"),
    "SpikeGPT_78_Scratch": os.path.join(base_dir, "78_Scratch", "metrics_report.txt"),
    "SpikeGPT_220_Scratch": os.path.join(base_dir, "220_Scratch", "metrics_report.txt"),
    "SpikeGPT_220_Finetune": os.path.join(base_dir, "220_Finetune", "metrics_report.txt")
}

results = {}
for name, path in models.items():
    results[name] = parse_metrics(path)

# Placeholders for GPT2
results["GPT2_Small"] = {
    'Valid JSON (%)': 98.5, # Placeholder
    'Intent Accuracy (%)': 95.0, # Placeholder
    'Args Exact Match (%)': 88.0, # Placeholder
    'Precision': 0.955,
    'Recall': 0.950,
    'F1 Score': 0.952
}

results["GPT2_Medium"] = {
    'Valid JSON (%)': 99.2, # Placeholder
    'Intent Accuracy (%)': 97.5, # Placeholder
    'Args Exact Match (%)': 93.0, # Placeholder
    'Precision': 0.978,
    'Recall': 0.975,
    'F1 Score': 0.976
}

df = pd.DataFrame(results).T
df = df.reset_index().rename(columns={'index': 'Model'})
display(df)

In [ ]:
def plot_comparison(df_subset, title):
    df_melted = df_subset.melt(id_vars='Model', var_name='Metric', value_name='Score')
    
    pct_metrics = ['Valid JSON (%)', 'Intent Accuracy (%)', 'Args Exact Match (%)']
    score_metrics = ['Precision', 'Recall', 'F1 Score']
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    sns.barplot(data=df_melted[df_melted['Metric'].isin(pct_metrics)], x='Metric', y='Score', hue='Model', ax=axes[0])
    axes[0].set_title(f'{title} - Percentages')
    axes[0].set_ylim(0, 110)
    for container in axes[0].containers:
        axes[0].bar_label(container, fmt='%.1f')
        
    sns.barplot(data=df_melted[df_melted['Metric'].isin(score_metrics)], x='Metric', y='Score', hue='Model', ax=axes[1])
    axes[1].set_title(f'{title} - Scores')
    axes[1].set_ylim(0, 1.1)
    for container in axes[1].containers:
        axes[1].bar_label(container, fmt='%.3f')
        
    plt.tight_layout()
    plt.show()

## 1. Chứng minh HeadQK cải thiện kết quả
So sánh SpikeGPT train from scratch epoch 78 có HeadQK=256 và không có HeadQK.

In [ ]:
df_headqk = df[df['Model'].isin(['SpikeGPT_78_NoHeadQK', 'SpikeGPT_78_Scratch'])]
plot_comparison(df_headqk, "SpikeGPT Epoch 78: HeadQK (256) vs No HeadQK")

## 2. Chứng minh Finetune từ Pretrain cải thiện kết quả
So sánh SpikeGPT train from scratch epoch 220 và SpikeGPT finetune từ pretrain 216M epoch 220 (Cả 2 đều có HeadQK=256).

In [ ]:
df_finetune = df[df['Model'].isin(['SpikeGPT_220_Scratch', 'SpikeGPT_220_Finetune'])]
plot_comparison(df_finetune, "SpikeGPT Epoch 220: Finetune vs Scratch")

## 3. Đối chứng với mô hình Transformer chuẩn (GPT-2)
So sánh phiên bản tốt nhất của SpikeGPT (Finetune Epoch 220) với mô hình GPT2-Small và GPT2-Medium để thấy khoảng cách về hiệu năng giữa SNN và Transformer thuần túy.

In [ ]:
df_gpt2 = df[df['Model'].isin(['SpikeGPT_220_Finetune', 'GPT2_Small', 'GPT2_Medium'])]
plot_comparison(df_gpt2, "SpikeGPT (Best) vs GPT2")